# Neurotransmitter Probability Variance across Drosophila Neuropils

This projects aims to answer - how do neurotransmitter probability distributions vary across neuropils in Drosophila?
The datasets should be downloaded into the data directory following the instructions on GitHub. 

## Set up environment

In [1]:
%load_ext autoreload 
%autoreload 2

# Import external libraries
from IPython.core.display import HTML
import pyvista as pv
from dask.distributed import Client

# Import core python libraries
import os

# Import local scripts (brainz.py, pipeline.py, plotting.py, 
# preprocess.py, util.py)
from scripts import *

In [ ]:
# Set up in-line 3D plot rendering
pv.set_jupyter_backend("client")
HTML("""
<style>
.output_svg {
    display: table-cell;
    text-align: center;
    vertical-align: middle;
}
</style>
""")

In [2]:
# Set globals
OUTDIR = os.path.join(os.path.dirname(__name__), "results", "notebook")
if not os.path.exists(OUTDIR): os.mkdir(OUTDIR)
MINSIZE = 30 # Minimum number of nodes in a cluster/community

In [3]:
allow_bytes = pipeline.get_ram_allowance()
client = pipeline.start_dask(num_cores=os.cpu_count(), allow_bytes=allow_bytes)
preprocess.run() # ~10 mins first run

15.8 GB available on machine; allowing 7.9 GB

Dask Dashboard at http://192.168.0.209:8787/status

Notice: this may take about 10 minutes if this is the first time running preprocess.py. 
10:08:49 Converting C:\Users\cielb\Documents\coding\_NTPDVND\data\proofread_connections_783.feather to parquet file ...


Converting ...: 100%|████████████████████████████████████████████████████████████████| 258/258 [01:05<00:00,  3.92it/s]


10:09:55 C:\Users\cielb\Documents\coding\_NTPDVND\data\proofread_connections_783.feather converted to parquet file
10:09:55 Converting C:\Users\cielb\Documents\coding\_NTPDVND\data\flywire_synapses_783.feather to parquet file ...


Converting ...: 100%|██████████████████████████████████████████████████████████████| 1985/1985 [12:21<00:00,  2.68it/s]


10:22:17 C:\Users\cielb\Documents\coding\_NTPDVND\data\flywire_synapses_783.feather converted to parquet file
10:22:18 Generating test parquet files ...


2026-05-29 10:22:35,010 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle ad8d8e37d382a8c1a67cc8842d6c5f67 initialized by task ('shuffle-transfer-ad8d8e37d382a8c1a67cc8842d6c5f67', 4) executed on worker inproc://192.168.0.209/15776/4
2026-05-29 10:22:35,335 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 70430cb72b4d6c0e2602d8fbbd51fee2 initialized by task ('shuffle-transfer-70430cb72b4d6c0e2602d8fbbd51fee2', 9) executed on worker inproc://192.168.0.209/15776/4
2026-05-29 10:22:36,807 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle ad8d8e37d382a8c1a67cc8842d6c5f67 deactivated due to stimulus 'task-finished-1780006956.7968988'
2026-05-29 10:22:37,156 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 70430cb72b4d6c0e2602d8fbbd51fee2 deactivated due to stimulus 'task-finished-1780006957.1534345'
2026-05-29 10:22:38,289 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 381517ca7b0e23f005608df466e84783 initialized by task ('shuffle-trans

10:22:58 Generated test parquet files

10:22:58 Preprocessing complete!


2026-05-29 10:27:15,554 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 47be206ffa86b98ae82ced4802fb1cd9 initialized by task ('shuffle-transfer-47be206ffa86b98ae82ced4802fb1cd9', 0) executed on worker inproc://192.168.0.209/15776/4
2026-05-29 10:27:15,653 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 47be206ffa86b98ae82ced4802fb1cd9 deactivated due to stimulus 'task-finished-1780007235.6521046'
2026-05-29 10:27:33,319 - distributed.worker.memory - WARNING - Worker is at 80% memory usage. Pausing worker.  Process memory: 6.36 GiB -- Worker memory limit: 7.90 GiB
2026-05-29 10:27:36,941 - distributed.worker.memory - WARNING - Worker is at 77% memory usage. Resuming worker. Process memory: 6.16 GiB -- Worker memory limit: 7.90 GiB
2026-05-29 10:28:09,357 - distributed.worker.memory - WARNING - Worker is at 92% memory usage. Pausing worker.  Process memory: 7.33 GiB -- Worker memory limit: 7.90 GiB
2026-05-29 10:28:31,011 - distributed.worker.memory - WARNING - Wo

## Preview - Can Nodule Neurons be Isolated from Linker Neurons?

Here I visualise the initial output generated from HDBSCAN clustering on xyz coordinates on a small sample of the drosophila connectome. The aim is to ensure the clustering parameters assign cluster IDs to groups in a way that approximates real nodules. As the 3D plot shows, the Drosophila connectome cannot be segregated into 'nodules' using this approach. Indeed, most synapses are equidistant from each other in 3D space. To isolate 'nodule'-like structures and generate visualisations such as those seen in the literature, filtering by cell type is required, however, cell type annotations are not present in this dataset. Interestingly, there do appear to be some patches of yellow, blue, and pink, indicating there may still be some differentiation in neurotransmitter probabilities across neuropils. The remainder of this analysis will focus on comparing neurotransmitter probabilities across neuropils using the unclustered dataframe.

In [4]:
connectome = pipeline.load_connectome("data/tiny.parquet")
connectome = pipeline.normalise_nt_probs(connectome)
connectome = pipeline.attach_synapse_coords(connectome)
condensed = pipeline.condense(connectome)
condensed = util.do_hdbscan(condensed, MINSIZE)
clustered = pipeline.extend(condensed, connectome)

10:27:15 Loading connectome ...
10:27:15 Connectome loaded
10:27:15 Normalising neurotransmitter probabilities ...
10:27:16 Neurotransmitter probabilities normalised
10:27:16 Attaching coordinates ...
10:35:18 Coordinates attached
10:35:19 Condensing synapses to neural connections ...
10:35:20 Condensed synapse coordinates to neural connections


NameError: name 'util' is not defined

In [ ]:
# Plot 500,000 points
plotter = brainz.get_plotter(clustered, "hdbscan_id")
brainz.save(plotter, OUTDIR, _id="clusterd_brain_map")
print("Saved brain map!")
plotter.show()

In [ ]:
plotter.close() # Free memory

## Visualise Neurotransmitter Probability Distributions

### Overall Neurotransmitter Probability Distributions

In [ ]:
# Sample ~1 million points to speed up render (using matplotlib)
#sample = util.downsample(connectome, 1_000_000)
sample = connectome
filename = os.path.join(OUTDIR, "overall_distribution.svg")
plotting.plot_overall_distribution(sample, "Full Dataset", filename)

### Get Neuropil Summary Statistics

In [ ]:
neuropils = pipeline.get_neuropil_summary_stats(connectome)
neuropils.head(10)

### Neuropil Probability Distributions with respect to Size (Number of Synapses)

In [ ]:
filename = os.path.join(OUTDIR, "mean_neuropil_probs.svg")
plotting.plot_mean_nt_probs_by_neuropil_size(neuropils, filename)

### Neurotransmitter Probability Variance by Neuropil Size

In [ ]:
filename = os.path.join(OUTDIR, "neuropil_nt_prob_variance_by_size.svg")
plotting.plot_variance_by_neuropil_size(neuropils, filename)

### Neuropil Probability Distributions

In [ ]:
filename = os.path.join(OUTDIR, "neuropil_probability_distributions.svg")
plotting.plot_hex_per_neuropil(connectome, filename)

## Statistical Analysis

I want to know whether neurotransmitter probabilities are different between neuropils. Because probabilities are a form of compositional data (the sum of the variables is 1, and each variable is bounded between 0 and 1), the Dirichlet distribution is suitable. This statistical analysis is a simple one based on average synapse probabilities within neuropils. A more statistically robust method would involve using the 'other' column to calculate the ranges of excitatory and inhibitory probabilities for each synaptic connection, then running a statistical analysis that can handle comparing range values within a bounded interval, e.g., Bayesian Dirichlet regression with interval priors. The Dirichlet function I used can only operate on points, not ranges. There does not appear to be an out-of-box Dirichlet regression function in any Python libraries, so I will interface with the R DirichletReg package via rpy2.

In [ ]:
fitted, model_summary, null_summary, anova_result = do_stats.run(connectome)